# 01 — Data Loading & Exploration

Loads 5 years of daily OHLCV for the top-50 S&P 500 stocks + SPY, caches to
`data/raw/ohlcv_daily.parquet`, resamples to weekly / monthly / annual, and
validates adjusted-price handling.

**yfinance ≥ 1.7 column semantics** (`auto_adjust=False`):
- `close` — split-adjusted only (adjusts for stock splits, NOT dividends)
- `adj_close` — fully adjusted (splits AND dividends)

Downstream return calculations should use `adj_close`. Dividends are real
economic returns; using `close` understates total return for income-paying
stocks (e.g. XOM yields ~4 %/yr, so `close`-only cumulative returns drift
~25 % low over 5 years).

In [1]:
import logging
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

# Make src importable when running from the notebooks/week1/ directory
repo_root = Path().resolve().parents[1]
cache_dir = str(repo_root / "data" / "raw")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.universe import END_DATE, SP500_TOP50, START_DATE, UNIVERSE
from src.data_utils import load_ohlcv, resample_ohlcv

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
matplotlib.use("Agg")

print(f"Universe: {len(UNIVERSE)} tickers  ({len(SP500_TOP50)} S&P 500 + SPY)")
print(f"Date range: {START_DATE} → {END_DATE}")

Universe: 51 tickers  (50 S&P 500 + SPY)
Date range: 2020-08-30 → 2025-08-30


## 1. Load daily data (cached after first run)

In [2]:
import time

t0 = time.time()
daily = load_ohlcv(UNIVERSE, start=START_DATE, end=END_DATE, cache_dir=cache_dir)
elapsed = time.time() - t0

print(f"Loaded in {elapsed:.1f}s")
print(f"Shape: {daily.shape}")
print(f"Columns: {list(daily.columns)}")
daily.head(3)

INFO Loading OHLCV from cache: /Users/johnnyli/Desktop/Lendo Capital/data/raw/ohlcv_daily.parquet


Loaded in 0.0s
Shape: (64035, 8)
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


,date,ticker,open,high,low,close,adj_close,volume
0,2020-08-31,AAPL,127.580002,131.000000,126.000000,129.039993,125.057526,225702700.0
1,2020-08-31,ABBV,94.120003,96.010002,94.010002,95.769997,75.952591,9932300.0
2,2020-08-31,ACN,243.300003,243.539993,239.100006,239.929993,217.961899,2001100.0


In [3]:
# Confirm cache: second call should be near-instant (no API hit)
t1 = time.time()
daily2 = load_ohlcv(UNIVERSE, start=START_DATE, end=END_DATE, cache_dir=cache_dir)
cache_elapsed = time.time() - t1
assert daily2.shape == daily.shape
print(f"Second call (cache): {cache_elapsed:.2f}s  ← should be << first call ({elapsed:.1f}s)")

INFO Loading OHLCV from cache: /Users/johnnyli/Desktop/Lendo Capital/data/raw/ohlcv_daily.parquet


Second call (cache): 0.00s  ← should be << first call (0.0s)


## 2. Daily observations per ticker

In [4]:
obs_per_ticker = (
    daily.groupby("ticker")["date"]
    .agg(["count", "min", "max"])
    .rename(columns={"count": "daily_obs", "min": "first_date", "max": "last_date"})
    .sort_values("daily_obs")
)

max_obs = obs_per_ticker["daily_obs"].max()
print(f"Max daily observations (full 5-yr tickers): {max_obs}")
print(f"Date range in data: {daily['date'].min().date()} → {daily['date'].max().date()}")

# Flag any ticker with fewer rows than the full-coverage count — catches late
# IPOs whose pre-listing NaN rows were correctly dropped (e.g. PLTR).
flagged = obs_per_ticker[obs_per_ticker["daily_obs"] < max_obs].copy()
flagged["missing_days"] = max_obs - flagged["daily_obs"]
if flagged.empty:
    print("All tickers have full 5-year coverage.")
else:
    print("\nTickers with fewer observations than full coverage (late IPO / gap):")
    print(flagged.to_string())

obs_per_ticker

Max daily observations (full 5-yr tickers): 1256
Date range in data: 2020-08-31 → 2025-08-29

Tickers with fewer observations than full coverage (late IPO / gap):
        daily_obs first_date  last_date  missing_days
ticker                                               
PLTR         1235 2020-09-30 2025-08-29            21


,daily_obs,first_date,last_date
ticker,,,
PLTR,1235,2020-09-30,2025-08-29
MA,1256,2020-08-31,2025-08-29
META,1256,2020-08-31,2025-08-29
MRK,1256,2020-08-31,2025-08-29
MSFT,1256,2020-08-31,2025-08-29
NFLX,1256,2020-08-31,2025-08-29
NOW,1256,2020-08-31,2025-08-29
NVDA,1256,2020-08-31,2025-08-29
ORCL,1256,2020-08-31,2025-08-29


## 3. Summary statistics — daily frequency

In [5]:
print("=== Daily summary statistics ===")
daily[["open", "high", "low", "close", "adj_close", "volume"]].describe().round(2)

=== Daily summary statistics ===


,open,high,low,close,adj_close,volume
count,64035.00,64035.00,64035.00,64035.00,64035.00,6.403500e+04
mean,219.05,221.37,216.67,219.08,210.18,2.623336e+07
std,166.42,167.99,164.75,166.42,163.82,6.678878e+07
min,5.98,6.17,5.92,6.00,6.00,6.720000e+04
25%,96.89,97.93,95.71,96.92,90.09,3.357650e+06
50%,165.74,167.38,163.97,165.75,155.46,7.308900e+06
75%,310.08,313.44,306.87,310.42,296.15,2.207080e+07
max,1076.48,1078.23,1068.01,1076.86,1067.74,1.543911e+09


## 4. Resample to weekly / monthly / annual

In [6]:
weekly  = resample_ohlcv(daily, "W")
monthly = resample_ohlcv(daily, "ME")
annual  = resample_ohlcv(daily, "YE")

for label, df in [("Daily", daily), ("Weekly", weekly), ("Monthly", monthly), ("Annual", annual)]:
    n_tickers = df["ticker"].nunique()
    rows = len(df)
    print(f"{label:8s}: {rows:7,d} rows across {n_tickers} tickers")

Daily   :  64,035 rows across 51 tickers
Weekly  :  13,307 rows across 51 tickers
Monthly :   3,110 rows across 51 tickers
Annual  :     306 rows across 51 tickers


In [7]:
print("=== Weekly summary statistics ===")
weekly[["open", "high", "low", "close", "adj_close", "volume"]].describe().round(2)

=== Weekly summary statistics ===


,open,high,low,close,adj_close,volume
count,13307.00,13307.00,13307.00,13307.00,13307.00,1.330700e+04
mean,218.79,224.95,213.03,219.40,210.51,1.262383e+08
std,166.19,170.46,162.18,166.62,164.03,3.121940e+08
min,6.15,6.53,5.92,6.29,6.29,2.778100e+06
25%,96.79,99.64,94.16,97.17,90.20,1.673443e+07
50%,165.66,169.94,161.50,165.86,155.60,3.606450e+07
75%,310.00,318.68,301.99,310.65,296.55,1.075732e+08
max,1069.21,1078.23,1046.00,1071.85,1062.77,4.308297e+09


In [8]:
print("=== Monthly summary statistics ===")
monthly[["open", "high", "low", "close", "adj_close", "volume"]].describe().round(2)

=== Monthly summary statistics ===


,open,high,low,close,adj_close,volume
count,3110.00,3110.00,3110.00,3110.00,3110.00,3.110000e+03
mean,217.23,230.98,204.70,219.47,210.60,5.401457e+08
std,165.42,174.77,156.33,167.20,164.65,1.312974e+09
min,6.58,7.82,5.92,6.42,6.42,8.194920e+05
25%,96.21,101.96,89.31,97.16,90.26,7.348582e+07
50%,164.47,174.97,155.87,165.32,156.82,1.553416e+08
75%,307.51,328.19,288.61,310.95,296.04,4.724388e+08
max,1051.74,1078.23,983.00,1048.61,1039.73,1.585655e+10


In [9]:
print("=== Annual summary statistics ===")
annual[["open", "high", "low", "close", "adj_close", "volume"]].describe().round(2)

=== Annual summary statistics ===


,open,high,low,close,adj_close,volume
count,306.00,306.00,306.00,306.00,306.00,3.060000e+02
mean,207.58,261.74,174.28,232.48,224.16,5.489716e+09
std,158.44,196.05,135.26,175.68,173.59,1.367278e+10
min,6.58,14.73,5.92,6.42,6.42,1.055387e+08
25%,86.39,114.79,73.93,100.99,94.10,7.059190e+08
50%,157.08,192.39,133.88,170.28,160.89,1.619686e+09
75%,300.08,368.74,242.91,335.82,324.89,4.753665e+09
max,915.00,1078.23,871.71,943.32,937.85,1.363340e+11


## 5. Spot-check resampling correctness (AAPL, January 2024)

In [10]:
aapl_daily   = daily[(daily["ticker"] == "AAPL") & (daily["date"].dt.to_period("M") == "2024-01")]
aapl_monthly = monthly[(monthly["ticker"] == "AAPL") & (monthly["date"].dt.to_period("M") == "2024-01")]

print("AAPL daily bars for 2024-01:")
print(aapl_daily[["date", "open", "high", "low", "close", "volume"]].to_string(index=False))
print()
print("AAPL monthly bar for 2024-01 (resampled):")
print(aapl_monthly[["date", "open", "high", "low", "close", "volume"]].to_string(index=False))
print()

exp_open  = aapl_daily["open"].iloc[0]
exp_high  = aapl_daily["high"].max()
exp_low   = aapl_daily["low"].min()
exp_close = aapl_daily["close"].iloc[-1]
exp_vol   = aapl_daily["volume"].sum()

row = aapl_monthly.iloc[0]
assert abs(row["open"]   - exp_open)  < 0.01, f"open mismatch: {row['open']} vs {exp_open}"
assert abs(row["high"]   - exp_high)  < 0.01, f"high mismatch"
assert abs(row["low"]    - exp_low)   < 0.01, f"low mismatch"
assert abs(row["close"]  - exp_close) < 0.01, f"close mismatch"
assert abs(row["volume"] - exp_vol)   < 1,    f"volume mismatch"
print("All OHLCV aggregation rules verified for AAPL 2024-01.")

AAPL daily bars for 2024-01:
      date       open       high        low      close     volume
2024-01-02 187.149994 188.440002 183.889999 185.639999 82488700.0
2024-01-03 184.220001 185.880005 183.429993 184.250000 58414500.0
2024-01-04 182.149994 183.089996 180.880005 181.910004 71983600.0
2024-01-05 181.990005 182.759995 180.169998 181.179993 62379700.0
2024-01-08 182.089996 185.600006 181.500000 185.559998 59144500.0
2024-01-09 183.919998 185.149994 182.729996 185.139999 42841800.0
2024-01-10 184.350006 186.399994 183.919998 186.190002 46792900.0
2024-01-11 186.539993 187.050003 183.619995 185.589996 49128400.0
2024-01-12 186.059998 186.740005 185.190002 185.919998 40477800.0
2024-01-16 182.160004 184.259995 180.929993 183.630005 65603000.0
2024-01-17 181.270004 182.929993 180.300003 182.679993 47317400.0
2024-01-18 186.089996 189.139999 185.830002 188.630005 78005800.0
2024-01-19 189.330002 191.949997 188.820007 191.559998 68903000.0
2024-01-22 192.300003 195.330002 192.259995 193

## 6. Top 10 / bottom 10 tickers by average daily volume

In [11]:
avg_vol = (
    daily.groupby("ticker")["volume"]
    .mean()
    .sort_values(ascending=False)
    .rename("avg_daily_volume")
)

print("Top 10 by average daily volume:")
print(avg_vol.head(10).apply(lambda x: f"{x:,.0f}").to_string())
print()
print("Bottom 10 by average daily volume:")
print(avg_vol.tail(10).apply(lambda x: f"{x:,.0f}").to_string())

Top 10 by average daily volume:
ticker
NVDA     411,588,978
TSLA     105,033,777
SPY       75,735,141
AAPL      75,560,944
AMD       60,971,570
AMZN      60,954,571
PLTR      60,080,271
NFLX      58,443,992
BAC       45,675,532
GOOGL     32,203,651

Bottom 10 by average daily volume:
ticker
MA      3,136,162
AMGN    2,702,565
GS      2,524,813
ACN     2,447,542
COST    2,159,780
LIN     1,885,099
ISRG    1,760,278
TMO     1,646,758
INTU    1,534,539
SPGI    1,520,914


## 7. Adjustment sanity check — dividend divergence (XOM) + split continuity (NVDA)

**yfinance ≥ 1.7 behavior with `auto_adjust=False`:**
- `close` is already split-adjusted across the entire history. There is NO
  price discontinuity at a split date in `close`.
- `adj_close` additionally removes dividend effects, so `close > adj_close`
  for dividend-paying stocks and the gap widens over time as dividends accrue.

**Check A:** XOM (Exxon Mobil) — high-yield stock (~4 %/yr). Over 5 years the
cumulative dividend adjustment should make `adj_close` noticeably lower than
`close`, growing wider from left to right.

**Check B:** NVDA (10-for-1 split on 2024-06-10) — confirm the price series
is continuous (no 10x jump in `close` at the split date), proving the
split adjustment is applied correctly.

In [12]:
# Check A: XOM dividend-driven divergence
xom = daily[daily["ticker"] == "XOM"].copy()

# Sample quarterly to see the widening gap
xom_q = (
    xom.set_index("date")[["close", "adj_close"]]
    .resample("QE")
    .last()
    .assign(ratio=lambda d: (d["close"] / d["adj_close"]).round(4))
)
print("XOM close vs adj_close (quarterly snapshots):")
print(xom_q.to_string())
print()

earliest_ratio = xom_q["ratio"].iloc[0]
latest_ratio   = xom_q["ratio"].iloc[-1]
print(f"Earliest ratio (close/adj_close): {earliest_ratio:.4f}")
print(f"Latest  ratio (close/adj_close): {latest_ratio:.4f}")
print(f"Implied cumulative dividend adjustment: {(earliest_ratio - 1)*100:.1f}%")

assert earliest_ratio > 1.10, "Expected close > adj_close by 10%+ over 5 years for XOM"
assert latest_ratio   > 1.00, "Expected close >= adj_close at all times"
print("\nCheck A PASSED — adj_close correctly removes cumulative dividend income.")

XOM close vs adj_close (quarterly snapshots):
                 close   adj_close   ratio
date                                      
2020-09-30   34.330002   26.858463  1.2782
2020-12-31   41.220001   33.026981  1.2481
2021-03-31   55.830002   45.492718  1.2272
2021-06-30   63.080002   52.149124  1.2096
2021-09-30   58.820000   49.363338  1.1916
2021-12-31   61.189999   52.042442  1.1758
2022-03-31   82.589996   71.021980  1.1629
2022-06-30   85.639999   74.399139  1.1511
2022-09-30   87.309998   76.586914  1.1400
2022-12-31  110.300003   97.532272  1.1309
2023-03-31  109.660004   97.712509  1.1223
2023-06-30  107.250000   96.394310  1.1126
2023-09-30  117.580002  106.545036  1.1036
2023-12-31   99.980003   91.425262  1.0936
2024-03-31  116.239998  107.281837  1.0835
2024-06-30  115.120003  107.111137  1.0748
2024-09-30  117.220001  109.943108  1.0662
2024-12-31  107.570000  101.721207  1.0575
2025-03-31  118.930000  113.469498  1.0481
2025-06-30  107.800003  103.797798  1.0386
2025-09-

In [13]:
# Check B: NVDA price continuity across 10-for-1 split on 2024-06-10
nvda = daily[daily["ticker"] == "NVDA"].copy()
split_date = pd.Timestamp("2024-06-10")

window = nvda[
    (nvda["date"] >= split_date - pd.Timedelta(days=14)) &
    (nvda["date"] <= split_date + pd.Timedelta(days=14))
][["date", "close", "adj_close"]].copy()
window["day_pct_chg"] = window["close"].pct_change().mul(100).round(2)

print("NVDA close around the 2024-06-10 split (split-adjusted, so no gap):")
print(window.to_string(index=False))

# The close at and around split_date should NOT jump by ~900% (which would
# indicate an un-adjusted series).  Max single-day move should be < 25%.
max_move = window["day_pct_chg"].abs().max()
assert max_move < 25, (
    f"Saw a {max_move:.1f}% single-day move — looks like an unadjusted series!"
)
print(f"\nMax single-day price move in window: {max_move:.2f}%  (< 25% — no split discontinuity)")
print("Check B PASSED — close is already split-adjusted; no discontinuity at split date.")

NVDA close around the 2024-06-10 split (split-adjusted, so no gap):
      date      close  adj_close  day_pct_chg
2024-05-28 113.901001 113.704231          NaN
2024-05-29 114.824997 114.626640         0.81
2024-05-30 110.500000 110.309105        -3.77
2024-05-31 109.633003 109.443611        -0.78
2024-06-03 115.000000 114.801338         4.90
2024-06-04 116.436996 116.235847         1.25
2024-06-05 122.440002 122.228477         5.16
2024-06-06 120.998001 120.788986        -1.18
2024-06-07 120.888000 120.679176        -0.09
2024-06-10 121.790001 121.579605         0.75
2024-06-11 120.910004 120.711052        -0.72
2024-06-12 125.199997 124.993973         3.55
2024-06-13 129.610001 129.396759         3.52
2024-06-14 131.880005 131.662979         1.75
2024-06-17 130.979996 130.764481        -0.68
2024-06-18 135.580002 135.356918         3.51
2024-06-20 130.779999 130.564789        -3.54
2024-06-21 126.570000 126.361725        -3.22
2024-06-24 118.110001 117.915649        -6.68

Max single-

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel A: XOM dividend divergence
axes[0].plot(xom["date"], xom["close"],     label="close (split-adj)", lw=1.2)
axes[0].plot(xom["date"], xom["adj_close"], label="adj_close (split+div adj)", lw=1.2, linestyle="--")
axes[0].set_title("XOM: close vs adj_close\n(gap = cumulative dividends)")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Price (USD)")
axes[0].legend()

# Panel B: NVDA split continuity
axes[1].plot(nvda["date"], nvda["close"], label="close", lw=1.2)
axes[1].axvline(split_date, color="red", linestyle=":", label="Split 2024-06-10")
axes[1].set_title("NVDA: close (split-adjusted, continuous)\n10-for-1 split on 2024-06-10")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Price (USD)")
axes[1].legend()

plt.tight_layout()
plt.savefig("adjustment_sanity_check.png", dpi=100)
plt.show()
print("Chart saved to adjustment_sanity_check.png")

Chart saved to adjustment_sanity_check.png


/var/folders/35/d26c1c9s0cnbfnfsmk98p0sw0000gn/T/ipykernel_4065/1963788591.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Observation counts per frequency — final summary

In [15]:
summary_rows = []
for label, df in [("Daily", daily), ("Weekly", weekly), ("Monthly", monthly), ("Annual", annual)]:
    ticker_counts = df.groupby("ticker").size()
    summary_rows.append({
        "Frequency": label,
        "Total rows": len(df),
        "Tickers": df["ticker"].nunique(),
        "Min obs/ticker": ticker_counts.min(),
        "Median obs/ticker": int(ticker_counts.median()),
        "Max obs/ticker": ticker_counts.max(),
    })

pd.DataFrame(summary_rows).set_index("Frequency")

,Total rows,Tickers,Min obs/ticker,Median obs/ticker,Max obs/ticker
Frequency,,,,,
Daily,64035,51,1235,1256,1256
Weekly,13307,51,257,261,261
Monthly,3110,51,60,61,61
Annual,306,51,6,6,6


---
**Task 2 quality-check summary**

- `load_ohlcv()` returns both `close` and `adj_close` as distinct columns.
- Second call to `load_ohlcv()` loads from cache (no API hit).
- Universe (50 stocks + SPY) and date range defined in `src/universe.py`.
- `resample_ohlcv()` aggregation rules verified against AAPL 2024-01 spot-check.
- Failed/missing tickers are logged, not silently dropped or crashing.
- Adjustment sanity check: XOM dividend divergence (Check A) and NVDA split
  continuity (Check B) both passed.
- Notebook runs top-to-bottom cleanly.